## 01 — Using Tippecanoe

`tippecanoe` is a command-line tool from Mapbox that converts GeoJSON into a vector tile pyramid.

One command replaces our entire Module 02 pipeline — and produces a smaller, faster output format. This notebook runs it, inspects the output, and maps every flag to a decision we already made by hand.

## Installation

On macOS with Homebrew:

```bash
brew install tippecanoe
```

On Linux (Ubuntu/Debian):

```bash
sudo apt-get install tippecanoe
```

Verify the install:

In [1]:
import subprocess
result = subprocess.run(["/opt/homebrew/bin/tippecanoe", "--version"], capture_output=True, text=True)
print(result.stdout or result.stderr)

tippecanoe v2.79.0



## Running Tippecanoe

The basic command:

```bash
tippecanoe \
  --output=railroads.pmtiles \
  --minimum-zoom=1 \
  --maximum-zoom=14 \
  --simplification=10 \
  --drop-densest-as-needed \
  --layer=railroads \
  ne_10m_railroads.geojson
```

Let's run it from Python and capture the output:

In [ ]:
from pathlib import Path
import subprocess
import time

input_file  = Path("../../data/ne_10m_railroads.geojson")
output_file = Path("../../data/railroads.pmtiles")

cmd = [
    "/opt/homebrew/bin/tippecanoe",
    f"--output={output_file}",
    "--force",                     # overwrite if exists
    "--minimum-zoom=1",
    "--maximum-zoom=14",
    "--simplification=10",         # Douglas-Peucker tolerance in tile pixels
    "--drop-densest-as-needed",    # drop features at low zoom if tile is too large
    "--layer=railroads",
    str(input_file),
]

t0 = time.perf_counter()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.perf_counter() - t0

print(result.stderr)   # tippecanoe writes progress to stderr
print(f"\nCompleted in {elapsed:.1f}s")

## Inspecting the Output

In [ ]:
size_mb = output_file.stat().st_size / 1_000_000
raw_mb  = input_file.stat().st_size  / 1_000_000

print(f"Input  (raw GeoJSON):  {raw_mb:.1f} MB")
print(f"Output (PMTiles):      {size_mb:.2f} MB")
print(f"Compression ratio:     {raw_mb / size_mb:.1f}×  smaller")

## Mapping Flags to Decisions We Already Made

Every `tippecanoe` flag corresponds to something we built or decided manually:

| tippecanoe flag | What it does | Our equivalent |
|-----------------|-------------|----------------|
| `--minimum-zoom` | First zoom level that gets tiles | Bottom of our LOD range |
| `--maximum-zoom` | Most detailed zoom level | Top of our LOD range |
| `--simplification=10` | D-P tolerance in tile pixels per zoom | Our epsilon per LOD level |
| `--drop-densest-as-needed` | Remove least-important features when tile is too large | Our `scalerank <= 4` coarse filter |
| `--layer=railroads` | Names the data layer in the tile | Our filename convention |

The flags we do NOT have to specify:
- Viewport culling — built into the tile addressing scheme
- Binary encoding — automatic (MVT format)
- Tile pyramid structure — automatic
- Spatial index — automatic (tiles ARE the index)
- Zoom-driven switching — automatic (client requests the right `{z}` tiles)


## Inspecting Tile Contents with sqlite3

PMTiles can be converted to `.mbtiles` (SQLite) for inspection. Or we can use the `pmtiles` CLI to peek at specific tiles.

Alternatively, inspect the metadata embedded in the PMTiles file:

In [1]:
# Use tippecanoe's companion tool to show metadata
result = subprocess.run(
    ["/opt/homebrew/bin/tile-join", "--no-tile-compression", "--if-matched",
     f"--output={output_file.with_suffix('.inspect.pmtiles')}",
     str(output_file)],
    capture_output=True, text=True
)

# Simpler: just show what pmtiles show gives us
result2 = subprocess.run(
    ["/opt/homebrew/bin/pmtiles", "show", str(output_file)],
    capture_output=True, text=True
)
print(result2.stdout or result2.stderr or "(install pmtiles CLI: pip install pmtiles)")

NameError: name 'subprocess' is not defined

## Viewing in ipyleaflet

ipyleaflet supports PMTiles through the `PMTilesLayer` (requires `ipyleaflet >= 0.18`).

For local files, we need to serve them via a local HTTP server or use `localtileserver`.

In [5]:
# Try loading with localtileserver if available
try:
    from localtileserver import TileClient, get_leaflet_tile_layer
    from ipyleaflet import Map

    client = TileClient(str(output_file))
    layer  = get_leaflet_tile_layer(client)
    m = Map(center=client.center(), zoom=client.default_zoom)
    m.add(layer)
    m
except ImportError:
    print("localtileserver not installed.")
    print("Install with: pip install localtileserver")
    print()
    print("Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.")
    print(f"File location: {output_file.resolve()}")

localtileserver not installed.
Install with: pip install localtileserver

Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.
File location: /Users/manoj/Downloads/4543-5993-Spatial-Data-main/Assignments/03-Data_Manager/data/railroads.pmtiles


## Exercise A

Run `tippecanoe` a second time with `--maximum-zoom=8` and compare the output file size.

Then answer: what did limiting the maximum zoom cost us in terms of user experience, and what did it save?

In [4]:
# Run tippecanoe with --maximum-zoom=8 and compare output size
# Your code here

output_file_z8 = Path("../../data/railroads_z8.pmtiles")

cmd_z8 = [
    "/opt/homebrew/bin/tippecanoe",
    f"--output={output_file_z8}",
    "--force",
    "--minimum-zoom=1",
    "--maximum-zoom=8",
    "--simplification=10",
    "--drop-densest-as-needed",
    "--layer=railroads",
    str(input_file),
]

t0 = time.perf_counter()

result = subprocess.run(cmd_z8, capture_output=True, text=True)

elapsed = time.perf_counter() - t0

print(result.stderr)
print(f"\nCompleted in {elapsed:.1f}s")

original_size = output_file.stat().st_size / 1_000_000
z8_size = output_file_z8.stat().st_size / 1_000_000

print(f"\nOriginal PMTiles (max zoom 14): {original_size:.2f} MB")
print(f"PMTiles with max zoom 8:        {z8_size:.2f} MB")

Read 0.00 million features
Read 0.01 million features
Read 0.02 million features
                              
Merging string pool           
Merging vertices              
Merging nodes                 
Merging index                 
Reordering geometry: 0% 
Reordering geometry: 1% 
Reordering geometry: 2% 
Reordering geometry: 3% 
Reordering geometry: 4% 
Reordering geometry: 5% 
Reordering geometry: 6% 
Reordering geometry: 7% 
Reordering geometry: 8% 
Reordering geometry: 9% 
Reordering geometry: 10% 
Reordering geometry: 11% 
Reordering geometry: 12% 
Reordering geometry: 13% 
Reordering geometry: 14% 
Reordering geometry: 15% 
Reordering geometry: 16% 
Reordering geometry: 17% 
Reordering geometry: 18% 
Reordering geometry: 19% 
Reordering geometry: 20% 
Reordering geometry: 21% 
Reordering geometry: 22% 
Reordering geometry: 23% 
Reordering geometry: 24% 
Reordering geometry: 25% 
Reordering geometry: 26% 
Reordering geometry: 27% 
Reordering geometry: 28% 
Reordering geometry:

Limiting the maximum zoom to 8 reduces the output file size because high-detail tiles for street and city-level zooms are no longer generated. This saves storage space, download bandwidth, and tile generation time. However, users lose fine geometric detail when zooming deeply into cities because the map no longer contains high-resolution tiles beyond zoom 8.

## Exercise B

The `--simplification=10` flag sets the tolerance in **tile pixels**, not degrees. At zoom 14, a tile covers roughly 2.4km × 2.4km in 4096 pixels — so one pixel ≈ 0.6m.

Calculate what `--simplification=10` means in meters at zoom levels 2, 5, 8, and 12. Compare these to the degree-based epsilon values we chose in Module 02.

In [3]:
# Calculate simplification tolerance in meters at different zoom levels
# Compare to our Module 02 epsilon choices
# Your code here

tile_size_m_z14 = 2400
pixels_per_tile = 4096
simplification_pixels = 10

meters_per_pixel_z14 = tile_size_m_z14 / pixels_per_tile
zooms = [2, 5, 8, 12]

print("Zoom | Meters per pixel | Simplification 10px")
print("-" * 50)

for z in zooms:
    meters_per_pixel = meters_per_pixel_z14 * (2 ** (14 - z))
    simplification_meters = meters_per_pixel * simplification_pixels
    print(z, "|", round(meters_per_pixel, 2), "|", round(simplification_meters, 2), "meters")


Zoom | Meters per pixel | Simplification 10px
--------------------------------------------------
2 | 2400.0 | 24000.0 meters
5 | 300.0 | 3000.0 meters
8 | 37.5 | 375.0 meters
12 | 2.34 | 23.44 meters


The same --simplification=10 value represents different real-world distances depending on zoom level. At low zoom, 10 pixels covers a much larger distance, so more simplification happens. At high zoom, 10 pixels covers a smaller distance, so more detail is preserved. This is better than our degree-based epsilon values because it adapts to each zoom level automatically.

## Check Your Understanding

We ran `tippecanoe` with `--drop-densest-as-needed`. This flag tells tippecanoe to automatically drop the least-important features when a tile would otherwise be too large.

How does tippecanoe decide which features are "least important"? And how does that compare to our manual `scalerank <= 4` filter? Which approach is more principled — and what are the tradeoffs of each?

---

Tippecanoe decides which features are least important based on tile density and feature importance during tile generation. When a tile would be too large, it drops features from the densest parts of the tile first to keep the tile readable and within size limits.

 Our manual scalerank <= 4 filter is simpler because it uses an existing attribute to remove less important features at coarse zoom levels. Tippecanoe's approach is more adaptive because it responds to actual tile density, but the manual scalerank filter is easier to understand and control.

 The tradeoff is that tippecanoe is more production-ready and automatic, while scalerank filtering is more transparent but less flexible.

## Next

In [02 — The Comparison](./02-The_Comparison.ipynb), we put both systems side by side and answer the final question: what did `tippecanoe` actually save us from?